# 📓 Notebook 3｜ROC 曲線（站 3）

> 對應講義 Part 3（5.5）。把分類閾值從最左掃到最右，記錄「假警報率 a」與
> 「命中率 1−β」，畫成曲線。曲線離對角線越遠＝特徵越能分開兩類。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

# 兩類特徵值（有重疊）
x1 = rng.normal(0, 1, 500)     # ω1
x2 = rng.normal(1.5, 1, 500)   # ω2
print('ω1 平均 0，ω2 平均 1.5（有重疊）')

### 掃閾值，算 a 與 1−β

閾值 t：小於 t 判 ω1，大於 t 判 ω2。對每個 t 記錄：
- 假警報率 a = P(判 ω2 | 其實 ω1) = ω1 中 > t 的比例
- 命中率 1−β = P(判 ω2 | 其實 ω2) = ω2 中 > t 的比例

In [ ]:
thresholds = np.linspace(x1.min(), x1.max(), 200)
a_list, hit_list = [], []
for t in thresholds:
    a = (x1 > t).mean()          # 假警報率
    hit = (x2 > t).mean()        # 命中率 1−β
    a_list.append(a); hit_list.append(hit)

plt.figure(figsize=(6, 6))
plt.plot(a_list, hit_list, color='#4f46e5', lw=2, label='ROC')
plt.plot([0, 1], [0, 1], '--', color='#94a3b8', label='對角線（亂猜）')
plt.xlabel('假警報率 a'); plt.ylabel('命中率 1−β')
plt.title('ROC 曲線'); plt.legend(); plt.grid(alpha=.3); plt.show()

### AUC：曲線下方面積

AUC 介於 0.5（亂猜）到 1（完美分離）。用梯形法算。

In [ ]:
# 注意：a_list 隨閾值遞增而遞減，trapz 前要先按 a 升冪排序，否則 AUC 會是負的
order = np.argsort(a_list)
auc = np.trapz(np.array(hit_list)[order], np.array(a_list)[order])
print(f'AUC = {auc:.3f}')

# 對照：兩類完全重疊 → AUC≈0.5；分很開 → AUC≈1
x1b = rng.normal(0, 1, 500); x2b = rng.normal(0, 1, 500)      # 完全重疊
x1c = rng.normal(0, 1, 500); x2c = rng.normal(4, 1, 500)      # 分很開
def auc(x1, x2):
    ts = np.linspace(min(x1.min(), x2.min()), max(x1.max(), x2.max()), 300)
    a = np.array([(x1 > t).mean() for t in ts]); h = np.array([(x2 > t).mean() for t in ts])
    o = np.argsort(a)
    return np.trapz(h[o], a[o])
print(f'完全重疊 AUC={auc(x1b, x2b):.3f}   分很開 AUC={auc(x1c, x2c):.3f}')

# ✏️ 練習：把 ω2 平均改成 0.5，看 AUC 如何下降
